> **Note**: This notebook has been upgraded to support general Waste Detection. It analyzes the full TACO taxonomy dynamically and includes advanced analytics for small objects and dataset corruption.

# 📊 Waste Detection — Dataset Exploration & Analysis (Notebook 02)

### Overview
Deep dive into the TACO dataset before conversion. This notebook performs statistical analysis, class distribution profiling, bounding box analysis, and data quality checks (corruption, duplicates).

### New Features Added
- Small-object detection percentage calculation
- Duplicate image detection (MD5 hashing)
- Extended CSV reporting
- Dynamic category handling (no hardcoded limits)

### Pipeline Position
```
NB 01 (Download) → [taco/raw/] → NB 02 (THIS) → [Reports] → NB 03 (Conversion)
```

## 1. Environment Setup

In [ ]:
!pip install -q pycocotools pandas matplotlib seaborn opencv-python pillow rich tqdm

In [ ]:
import os
import json
import random
import hashlib
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image
from pycocotools.coco import COCO
from tqdm.auto import tqdm
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()
sns.set_theme(style="whitegrid")

## 2. Load Dataset
We load the raw TACO annotations using the `pycocotools` API.

In [ ]:
# ==========================================
# Paths
# ==========================================
PROJECT_ROOT = Path('/content/drive/MyDrive/PlasticSense_AI')
TACO_DIR = PROJECT_ROOT / 'datasets' / 'taco'
IMAGES_DIR = TACO_DIR / 'images'
ANN_FILE = TACO_DIR / 'annotations' / 'annotations.json'

REPORTS_DIR = TACO_DIR / 'reports' / 'analysis'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# ==========================================
# Load COCO API
# ==========================================
if not ANN_FILE.exists():
    raise FileNotFoundError(f"Annotations missing at {ANN_FILE}. Run Notebook 01.")

console.print("[cyan]Loading COCO annotations...[/cyan]")
coco = COCO(str(ANN_FILE))

# Test loaded data
img_ids = coco.getImgIds()
cat_ids = coco.getCatIds()
ann_ids = coco.getAnnIds()

console.print(f"[green]✔ Loaded {len(img_ids)} images, {len(cat_ids)} categories, {len(ann_ids)} annotations[/green]")

## 3. Macro Statistics
High-level view of the dataset scale.

In [ ]:
# ==========================================
# Generate Macro Stats
# ==========================================
def generate_macro_stats(coco: COCO, images_dir: Path) -> dict:
    images = coco.loadImgs(coco.getImgIds())
    
    anns_per_img = [len(coco.getAnnIds(imgIds=img['id'])) for img in images]
    widths = [img['width'] for img in images]
    heights = [img['height'] for img in images]
    
    # Calculate storage size
    folder_size_bytes = 0
    if images_dir.exists():
        folder_size_bytes = sum(f.stat().st_size for f in images_dir.glob('*') if f.is_file())
    
    stats = {
        'Total Images': len(images),
        'Total Categories': len(coco.getCatIds()),
        'Total Annotations': len(coco.getAnnIds()),
        'Avg Annotations / Image': round(np.mean(anns_per_img), 2),
        'Max Annotations / Image': np.max(anns_per_img),
        'Min Annotations / Image': np.min(anns_per_img),
        'Dataset Size (MB)': round(folder_size_bytes / (1024 * 1024), 2),
        'Avg Image Width': round(np.mean(widths), 1),
        'Avg Image Height': round(np.mean(heights), 1),
    }
    
    # Save CSV
    pd.DataFrame([stats]).to_csv(REPORTS_DIR / 'macro_statistics.csv', index=False)
    
    # Print
    table = Table(title="Dataset Macro Summary", show_header=True)
    table.add_column("Metric", style="cyan")
    table.add_column("Value", justify="right")
    for k, v in stats.items():
        table.add_row(k, str(v))
    console.print(table)
    
    return stats

macro_stats = generate_macro_stats(coco, IMAGES_DIR)

## 4. Category Distribution Analysis
Identify class imbalance across all TACO categories.

In [ ]:
# ==========================================
# Category Analysis
# ==========================================
def analyze_categories(coco: COCO):
    cats = coco.loadCats(coco.getCatIds())
    cat_names = [c['name'] for c in cats]
    cat_ids = [c['id'] for c in cats]
    
    ann_counts = [len(coco.getAnnIds(catIds=cid)) for cid in cat_ids]
    
    df_cats = pd.DataFrame({'Category': cat_names, 'Count': ann_counts})
    df_cats = df_cats.sort_values('Count', ascending=False).reset_index(drop=True)
    df_cats['Percentage'] = (df_cats['Count'] / df_cats['Count'].sum()) * 100
    
    df_cats.to_csv(REPORTS_DIR / 'category_distribution.csv', index=False)
    
    # Visualizations
    # 1. Bar Chart (Top 25)
    plt.figure(figsize=(14, 8))
    sns.barplot(data=df_cats.head(25), x='Count', y='Category', palette='viridis')
    plt.title('Top 25 Categories by Annotation Count', fontsize=16)
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / 'category_barplot.png')
    plt.show()
    
    # Summary
    console.print(Panel.fit(
        f"[bold cyan]Category Summary[/bold cyan]\n\n"
        f"Total Categories: {len(df_cats)}\n"
        f"Top Category: {df_cats.iloc[0]['Category']} ({df_cats.iloc[0]['Count']} anns)\n"
        f"Bottom Category: {df_cats.iloc[-1]['Category']} ({df_cats.iloc[-1]['Count']} anns)\n"
        f"Empty Categories: {len(df_cats[df_cats['Count'] == 0])}"
    ))

analyze_categories(coco)

## 5. Bounding Box & Small Object Analysis
Analyze object sizes to determine the difficulty of detection (small objects are notoriously hard for YOLO).

In [ ]:
# ==========================================
# Bounding Box Analysis
# ==========================================
def analyze_bboxes(coco: COCO):
    anns = coco.loadAnns(coco.getAnnIds())
    
    box_data = []
    for ann in tqdm(anns, desc="Processing BBoxes"):
        if 'bbox' in ann and len(ann['bbox']) == 4:
            x, y, w, h = ann['bbox']
            img_info = coco.loadImgs(ann['image_id'])[0]
            img_area = img_info['width'] * img_info['height']
            box_area = w * h
            
            box_data.append({
                'id': ann['id'],
                'image_id': ann['image_id'],
                'category_id': ann['category_id'],
                'width': w,
                'height': h,
                'area': box_area,
                'relative_area': (box_area / img_area) * 100 if img_area > 0 else 0
            })
            
    df_boxes = pd.DataFrame(box_data)
    
    # Small object analysis (COCO defines small as area < 32^2 = 1024)
    # Alternatively, relative area < 1% of image
    small_coco = len(df_boxes[df_boxes['area'] < 1024])
    small_rel = len(df_boxes[df_boxes['relative_area'] < 1.0])
    total = len(df_boxes)
    
    stats = {
        'total_boxes': total,
        'avg_relative_area_pct': df_boxes['relative_area'].mean(),
        'small_boxes_coco_def': small_coco,
        'small_boxes_coco_pct': (small_coco / total) * 100,
        'small_boxes_rel_1pct': small_rel,
        'small_boxes_rel_pct': (small_rel / total) * 100,
    }
    
    pd.DataFrame([stats]).to_csv(REPORTS_DIR / 'bbox_statistics.csv', index=False)
    
    # Visualization
    plt.figure(figsize=(10, 6))
    sns.histplot(np.log10(df_boxes['relative_area'] + 1e-5), bins=50)
    plt.title('Bounding Box Relative Area Distribution (Log10 %)', fontsize=14)
    plt.xlabel('Log10(Relative Area %)')
    plt.savefig(REPORTS_DIR / 'bbox_area_dist.png')
    plt.show()
    
    console.print(Panel.fit(
        f"[bold cyan]Small Object Analysis[/bold cyan]\n\n"
        f"Avg Object Area: {stats['avg_relative_area_pct']:.2f}% of image\n"
        f"Small Objects (COCO definition < 32x32): {small_coco} ({stats['small_boxes_coco_pct']:.1f}%)\n"
        f"Small Objects (<1% of image): {small_rel} ({stats['small_boxes_rel_pct']:.1f}%)"
    ))

analyze_bboxes(coco)

## 6. Dataset Integrity Checks
Check for duplicate and corrupted images before converting to YOLO format.

In [ ]:
# ==========================================
# Integrity Checks
# ==========================================
def run_integrity_checks(coco: COCO, images_dir: Path):
    console.print("[cyan]Running integrity checks (duplicates & corruption)...[/cyan]")
    
    expected_imgs = coco.loadImgs(coco.getImgIds())
    
    stats = {
        'missing': 0,
        'corrupted': 0,
        'zero_byte': 0,
        'duplicates': 0
    }
    
    seen_hashes = set()
    
    for img in tqdm(expected_imgs, desc="Checking images"):
        img_path = images_dir / img['file_name'].split('/')[-1]
        
        if not img_path.exists():
            stats['missing'] += 1
            continue
            
        if img_path.stat().st_size == 0:
            stats['zero_byte'] += 1
            continue
            
        try:
            with Image.open(img_path) as im:
                im.verify()
        except Exception:
            stats['corrupted'] += 1
            continue
            
        # Hash check
        hasher = hashlib.md5()
        with open(img_path, 'rb') as f:
            hasher.update(f.read())
        h = hasher.hexdigest()
        
        if h in seen_hashes:
            stats['duplicates'] += 1
        else:
            seen_hashes.add(h)
            
    pd.DataFrame([stats]).to_csv(REPORTS_DIR / 'integrity_statistics.csv', index=False)
    
    console.print(Panel.fit(
        f"[bold cyan]Integrity Results[/bold cyan]\n\n"
        f"Missing: {stats['missing']}\n"
        f"Corrupted: {stats['corrupted']}\n"
        f"Zero Byte: {stats['zero_byte']}\n"
        f"Duplicates: {stats['duplicates']}"
    ))

if IMAGES_DIR.exists():
    run_integrity_checks(coco, IMAGES_DIR)
else:
    console.print("[yellow]⚠ Images directory not found. Skipping integrity checks.[/yellow]")

## 7. Summary

In [ ]:
console.print(Panel.fit(
    f"[bold green]Dataset Analysis Complete[/bold green]\n\n"
    f"All reports saved to:\n"
    f"{REPORTS_DIR}\n\n"
    f"[bold magenta]Next Notebook:[/bold magenta] 03_COCO_to_YOLO_Conversion.ipynb"
))